In [ ]:
# Cell 1: Library installation
!pip install wikipedia-api tokenizers torch --quiet

import wikipediaapi
import torch
import torch.nn as nn
from torch.nn import functional as F
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import os

# Hyperparameter settings
batch_size = 64      # How many sequences do we process in parallel?
block_size = 128     # What is the maximum context length?
max_iters = 2000     # Number of training steps
eval_interval = 200  # Interval for calculating the loss
learning_rate = 3e-4 # Learning rate
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 50
n_embd = 128         # Embedding dimension
n_head = 4           # Number of attention heads
n_layer = 4          # Number of Transformer blocks
dropout = 0.1

print(f"Usando o dispositivo: {device}")

# Cell 2: Wikipedia data scraping
wiki = wikipediaapi.Wikipedia(
    user_agent='MiniGPT_Bot/1.0 (contato@exemplo.com)',
    language='pt'
)

# List of articles to download and use for training
artigos = ["Inteligência_artificial", "Aprendizado_de_máquina", "Deep_learning", "Processamento_de_linguagem_natural", "Transformadores_(modelos_de_aprendizado_de_máquina)"]
texto_completo = ""

for titulo in artigos:
    page = wiki.page(titulo)
    if page.exists():
        texto_completo += page.text + "\n\n"

# Save to a local file
with open("wiki_treino.txt", "w", encoding="utf-8") as f:
    f.write(texto_completo)

print(f"Dataset criado com sucesso! Total de caracteres: {len(texto_completo)}")

# Cell 3: Tokenizer training
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]"], vocab_size=2000)
tokenizer.pre_tokenizer = Whitespace()

# Train using our text file
tokenizer.train(["wiki_treino.txt"], trainer)
vocab_size = tokenizer.get_vocab_size()

# Encode the entire text as numbers (IDs)
dados_codificados = tokenizer.encode(texto_completo).ids
dados_tensor = torch.tensor(dados_codificados, dtype=torch.long)

# Split into training and validation (90% / 10%)
n = int(0.9 * len(dados_tensor))
dados_treino = dados_tensor[:n]
dados_val = dados_tensor[n:]

print(f"Tamanho do vocabulário: {vocab_size}")

# Cell 4: Batch generator
def get_batch(split):
    data = dados_treino if split == 'train' else dados_val
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# Cell 5: GPT model structure (PyTorch)
class Head(nn.Module):
    """ A masked self-attention head (Causal Self-Attention) """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # Mask the future
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ Multiple attention heads in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    """ A simple linear layer followed by a nonlinearity """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] # Focus only on the last step
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# Cell 6: Run training
model = MiniGPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("Iniciando treinamento...")
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"Passo {iter}: Perda no treino {losses['train']:.4f}, Perda na validação {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Cell 7: Test the model (Text Generation)
prompt = "A inteligência artificial é"
contexto_ids = tokenizer.encode(prompt).ids
contexto_tensor = torch.tensor([contexto_ids], dtype=torch.long, device=device)

# Generate 50 new tokens
resultado_tokens = model.generate(contexto_tensor, max_new_tokens=50)[0].tolist()
texto_gerado = tokenizer.decode(resultado_tokens)

print("\n--- Texto Gerado pelo Mini-GPT ---")
print(texto_gerado)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.8/129.8 kB 13.2 MB/s eta 0:00:00
Usando o dispositivo: cuda
Dataset criado com sucesso! Total de caracteres: 88261
Tamanho do vocabulário: 2000
Iniciando treinamento...
Passo 0: Perda no treino 7.6165, Perda na validação 7.6188
Passo 200: Perda no treino 4.5503, Perda na validação 6.1164
Passo 400: Perda no treino 2.1676, Perda na validação 6.6444
Passo 600: Perda no treino 0.7434, Perda na validação 7.5200
Passo 800: Perda no treino 0.2747, Perda na validação 8.1144
Passo 1000: Perda no treino 0.1559, Perda na validação 8.6120
Passo 1200: Perda no treino 0.1169, Perda na validação 8.9653
Passo 1400: Perda no treino 0.0986, Perda na validação 9.3085
Passo 1600: Perda no treino 0.0868, Perda na validação 9.4993
Passo 1800: Perda no treino 0.0796, Perda na validação 9.7274
Passo 1999: Perda no treino 0.0744, Perda na validação 9.9557

--- Texto Gerado pelo Mini-